# Накопление охвата по рекламным блокам в течение периода
Пример выгрузки накопления охвата в (000) и % для рекламных блоков

## Описание задачи и условий расчета
- Регион: Россия 0+
- Период: 02.12.2024-15.12.2024
- ЦА: Все 10-45
- Место просмотра: Все места (Дом+Дача)
- Каналы: Национальная телекомпания ТНТ, СТС
- Блоки: Блок контент - Коммерческий; Блок распространение - Сетевой
- Статистики: AccumulativeReach(000), AccumulativeReach%

## Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к TVI API и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import re
import json
import datetime
import time
import pandas as pd
#import matplotlib.pyplot as plt
from IPython.display import JSON

from mediascope_api.core import net as mscore
from mediascope_api.mediavortex import tasks as cwt
from mediascope_api.mediavortex import catalogs as cwc

# Настраиваем отображение

# Включаем отображение всех колонок
pd.set_option('display.max_columns', None)
# Задаем максимальное количество выводимых строк. Раскомментируйте нужную строку
# 200 строк
# pd.set_option("display.max_rows", 200)
# Отображаем все строки. ВАЖНО! Отображение большого DataFrame требует много ресурсов
# pd.set_option("display.max_rows", None)

# Cоздаем объекты для работы с TVI API
mnet = mscore.MediascopeApiNetwork()
mtask = cwt.MediaVortexTask()
cats = cwc.MediaVortexCats()

## Справочники
Получим идентификаторы, которые будут использоваться для формирования условий расчета

In [ ]:
# Национальная телекомпания ТНТ, СТС
cats.get_tv_net(name=['ТНТ','СТС'])

In [ ]:
# Блок содержание - коммерческий
cats.get_tv_breaks_content(name=['Коммерческий'])

In [ ]:
# Блок распространение - сетевой
cats.get_tv_breaks_distribution(name=['Сетевой'])

## Формирование задания
Зададим условия расчета

In [ ]:
# Задаем период
# Период указывается в виде списка ('Начало', 'Конец') в формате 'YYYY-MM-DD'. Можно указать несколько периодов
date_filter = [('2024-12-02', '2024-12-15')]

# Задаем дни недели
weekday_filter = None

# Задаем тип дня
daytype_filter = None

# Задаем ЦА: Все 10-45
basedemo_filter = 'age >= 10 and age <= 45'

# Доп фильтр ЦА, нужен только в случае расчета отношения между ЦА, например, при расчете Affinity Index
targetdemo_filter = None

# Задаем место просмотра
location_filter=None

# Задаем каналы: Национальная телекомпания ТНТ, СТС
company_filter = 'tvNetId IN (83,11)'

# Фильтр программ
program_filter = None

# Фильтр блоков: Блок контент - Коммерческий; Блок распространение - Сетевой
break_filter = 'breaksContentType = C and breaksDistributionType IN (N)'

# Фильтр роликов
ad_filter = None

# Указываем список срезов
slices = [
    'researchDate', # Дата, обязательный атрибут!
    'tvCompanyName', # Телекомпания
    'breaksPrimeTimeStatusName', # Прайм/ОфПрайм блоков
    'breaksSpotId', # Блок ID выхода, обязательный атрибут! 
    'breaksStartTime', # Блок время начала
    'breaksFinishTime', # Блок время окончания
]

# Указываем список статистик для расчета
statistics = ['AccumulativeReach000', 'AccumulativeReachPer']

# Задаем условия сортировки: дата (от старых к новым), время начала (по возр.)
sortings = {'researchDate':'ASC', 'breaksStartTime':'ASC', 'breaksSpotId':'ASC'}

# Задаем опции расчета
options = {
    "kitId": 1 #TV Index Russia all
}

## Расчет задания

In [ ]:
# Формируем задание для API TV Index в формате JSON
task_json = mtask.build_simple_task(date_filter=date_filter, weekday_filter=weekday_filter, 
                                        daytype_filter=daytype_filter, company_filter=company_filter, 
                                        location_filter=location_filter, basedemo_filter=basedemo_filter, 
                                        targetdemo_filter=targetdemo_filter,program_filter=program_filter, 
                                        break_filter=break_filter, ad_filter=ad_filter, 
                                        slices=slices, statistics=statistics, sortings=sortings, options=options)

# Отправляем задание на расчет и ждем выполнения
task_simple = mtask.wait_task(mtask.send_simple_task(task_json))

# Получаем результат
df = mtask.result2table(mtask.get_result(task_simple))

In [ ]:
# Приводим порядок столбцов в соответствие с условиями расчета
df = df[slices+statistics]
df

## Сохраняем в Excel
По умолчанию файл сохраняется в папку `excel`

In [ ]:
df_info = mtask.task_builder.get_report_info()

with pd.ExcelWriter(mtask.task_builder.get_excel_filename('simple_breaks_02_accumulative_reach')) as writer:
    df.to_excel(writer, sheet_name='Report', index=True)
    df_info.to_excel(writer, sheet_name='Info', index=False)